# 07 · Modular RAG: router → specialised retrievers → reranker → generator
**Concept:** independent, swappable modules plus routing by question type.
**TAC scenario:** like assignment groups, route to the KB, release notes, case archive or product guides.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank
from rag_common import format_docs, doc_ids

## Module 1: specialised retrievers (one per source)

In [ ]:
Source = Literal["kb", "release_notes", "tickets", "guides"]
DOC_TYPE = {"kb": "kb", "release_notes": "release_note", "tickets": "ticket", "guides": "guide"}

retrievers = {s: vectorstore.as_retriever(search_kwargs={"k": 4, "filter": {"doc_type": t}}) for s, t in DOC_TYPE.items()}
retrievers["tickets"] = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "filter": {"doc_type": "ticket"}})  # tuned: tickets repeat a lot

## Module 2: router

In [ ]:
class Route(BaseModel):
    sources: list[Source] = Field(description="One or more sources to search")
    reason: str

router = model.with_structured_output(Route)
ROUTER_PROMPT = """Route this TAC question to the right knowledge sources:
- kb: known issues, symptoms, root causes, workarounds
- release_notes: which release fixes or contains something
- tickets: whether other customers saw it before and how it was resolved
- guides: how-to and configuration commands
Question: {q}"""

## Modules 3 and 4: reranker and generator (each swappable)

In [ ]:
modules = {
    "reranker": LLMListwiseRerank.from_llm(llm=model, top_n=4),
    "generator": ChatPromptTemplate.from_messages([
        ("system", "Answer ONLY from the context and cite [doc_id]. If it is not there, reply NOT_IN_KB."),
        ("human", "<context>\n{context}\n</context>\nQuestion: {q}"),
    ]) | model | StrOutputParser(),
}

def modular_rag(q: str, verbose: bool = True) -> str:
    route = router.invoke(ROUTER_PROMPT.format(q=q))
    docs = [d for s in route.sources for d in retrievers[s].invoke(q)]
    if modules["reranker"] is not None and len(docs) > 4:
        docs = modules["reranker"].compress_documents(docs, q)
    if verbose:
        print(f"🧭 route → {route.sources}  ({route.reason})\n📄 evidence → {doc_ids(docs)}")
    return modules["generator"].invoke({"context": format_docs(docs), "q": q})

## Run it: different questions, different routes

In [ ]:
for q in ["Why does best-effort traffic get crushed during voice bursts after upgrading the AR-900 to 9.2.1?",
          "Which Nimbus NF-300 release fixes the IKEv2 phase 2 resets when the responder picks a different DH group?",
          "Have other customers seen the HX-5000 stack keep changing masters on 7.3? How was it fixed?",
          "Which command shows transceiver diagnostics on the HX-5000?"]:
    print(f"\n❓ {q}")
    print("💬", modular_rag(q))

## Swap a module without touching the rest

In [ ]:
modules["reranker"] = None          # e.g. remove the reranker to save cost or latency
print(modular_rag("Which Orion OW-20 release added configuration backup validation?"))

---
**🔧 TAC takeaway:** Assignment groups for questions: each source has its own tuned retriever, the router sends work to the right group, and any module can be replaced without rebuilding the system.